# Bar Path Model Training

Training an model for bar path tracking using exercise data

In [16]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Load data and config

In [17]:
from utils import load_airtable_data, load_config, save_data, load_mongo_data

config = load_config()
#data = load_airtable_data()
data =load_mongo_data()
save_data(data)

## Train an exercise classifier

For each data column

- Extract features from the entire time window for each data set
- Train model to predict exercise based on these feature


#### To Do

- Add back FFT features
- Feature selection
- Feature normalization
- Plot classifier results

In [11]:
EXERCISE_CLASSIFIER = 'Exercise'

### Extract features and labels

In [23]:
from preprocessing import preprocess_data, extract_features, extract_labels

data = preprocess_data(data, config['CutoffFreq'], config['SampleRate'])
features = extract_features(data)
labels = extract_labels(data, EXERCISE_CLASSIFIER)

### Train model

In [24]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.2)

exercise_classifier = XGBClassifier()
exercise_classifier.fit(X_train, y_train)

print("Classifier type: ", EXERCISE_CLASSIFIER)
print("Training accuracy: ", exercise_classifier.score(X_train, y_train))
print("Testing accuracy: ", exercise_classifier.score(X_test, y_test))

exercise_predictions = exercise_classifier.predict(X_test)
report = classification_report(y_test, exercise_predictions, zero_division=0)
print(report)

Classifier type:  Exercise
Training accuracy:  1.0
Testing accuracy:  0.9454545454545454
              precision    recall  f1-score   support

           0       0.90      1.00      0.95         9
           2       1.00      0.75      0.86         4
           3       1.00      1.00      1.00         4
           5       0.50      1.00      0.67         1
           7       1.00      1.00      1.00         6
           8       0.75      1.00      0.86         3
           9       1.00      1.00      1.00         5
          11       1.00      0.82      0.90        11
          12       1.00      1.00      1.00         3
          13       1.00      1.00      1.00         5
          14       1.00      1.00      1.00         4

    accuracy                           0.95        55
   macro avg       0.92      0.96      0.93        55
weighted avg       0.96      0.95      0.95        55



### Save classifer, features, and labels

In [25]:
from utils import save_classifier, save_features, save_labels

save_classifier(exercise_classifier, EXERCISE_CLASSIFIER, report)
save_features(features, EXERCISE_CLASSIFIER)
save_labels(labels, EXERCISE_CLASSIFIER)

## Train a predictor for whether is in rep state or non rep state

For entries with a RepStartTime and RepEndTime
 - Extract windows for each axes
 - Label window with 0 or 1 depending on whether it is between RepStartTime and RepEndTime
 - Create a classifier for this data

#### To Do

- Add back in FFT features
- Feature selection
- Feature normalization
- Plot classifier results

In [26]:
WINDOW_CLASSIFIER = 'Window'

### Extract windowed features and labels

In [27]:
from preprocessing import extract_window_features, extract_window_labels

config = load_config()
window_features = extract_window_features(data, config['WindowLength'], config['WindowStride'])
window_labels = extract_window_labels(data, config['WindowLength'], config['WindowStride'], config['SampleRate'])

### Train model

In [28]:
X_train, X_test, y_train, y_test = train_test_split(window_features, window_labels, test_size=0.2)

window_classifier = XGBClassifier()
window_classifier.fit(X_train, y_train)

print("Classifier type: ", WINDOW_CLASSIFIER)
print("Training accuracy: ", window_classifier.score(X_train, y_train))
print("Testing accuracy: ", window_classifier.score(X_test, y_test))

y_pred = window_classifier.predict(X_test)
report = classification_report(y_test, y_pred, zero_division=0)
print(report)

Classifier type:  Window
Training accuracy:  1.0
Testing accuracy:  0.9604372722540343
              precision    recall  f1-score   support

           0       0.95      0.96      0.96      1720
           1       0.97      0.96      0.96      2122

    accuracy                           0.96      3842
   macro avg       0.96      0.96      0.96      3842
weighted avg       0.96      0.96      0.96      3842



### Save classifier, features and labels

In [29]:
save_classifier(window_classifier, WINDOW_CLASSIFIER, report)
save_features(window_features, WINDOW_CLASSIFIER)
save_labels(window_labels, WINDOW_CLASSIFIER)

### Save this notebook to a script

In [1]:
# Convert the notebook to a script
!jupyter nbconvert --to script BarPathModelTraining.ipynb --output train_models

[NbConvertApp] Converting notebook BarPathModelTraining.ipynb to script
[NbConvertApp] Writing 3757 bytes to train_models.py
